In [ ]:
import pandas as pd

In [ ]:
eavs_2022: pd.DataFrame = pd.read_excel(
    "../data/raw/2022_EAVS_for_Public_Release_V1.1.xlsx", engine="calamine"
)
eavs_2022.head()

In [ ]:
# These are the columns we care about. A3a is total voter registrations, and A3e is total rejected registrations.
columns = ["State_Full", "A3a", "A3e"]
eavs_2022 = eavs_2022.filter(items=columns)

# Remove all counties where there is no data for either of the columns
for col in columns:
    eavs_2022 = eavs_2022[eavs_2022[col] != "Data not available"]
    eavs_2022 = eavs_2022[eavs_2022[col] != "Does not apply"]

# Convert total voter registrations and total rejected registrations to numbers instead of strings.
eavs_2022["A3a"] = pd.to_numeric(eavs_2022["A3a"], dtype_backend="pyarrow")
eavs_2022["A3e"] = pd.to_numeric(eavs_2022["A3e"], dtype_backend="pyarrow")

# In order to get state data instead of county data, aggregate all remaining counties belonging to a state and summing their total registration and rejected registration values.
eavs_2022 = eavs_2022.groupby("State_Full").agg({"A3a": "sum", "A3e": "sum"})

# Rejected % is Total Rejected Registrations / Total Voter Registrations
eavs_2022["Reject_%"] = eavs_2022["A3e"] / eavs_2022["A3a"]

# Rename and filter columns to make similar to Looker Studio table
eavs_2022 = eavs_2022.rename(columns={"A3a": "Reject_Count"})
eavs_2022 = eavs_2022.filter(items=["Reject_Count", "Reject_%"])

eavs_2022.head()